# 3부. 데이터 전처리

장별 완성 예제. 학습용 가상 데이터이며 공식 시험 제출 형식이 아니다. examples 폴더에서 전체 실행한다.

In [ ]:
from pathlib import Path
import os
if not Path('lab.py').is_file() and Path('../lab.py').is_file():
    os.chdir('..')
assert Path('lab.py').is_file(), 'Run with the examples folder and data present'
get_ipython().run_line_magic('matplotlib', 'inline')


## 10장. 결측치 처리하기

삭제·대체의 기준과 학습 데이터에 맞춘 처리 방법을 익힌다.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer
from lab import split_task, NUM_CLASS

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
imputer = SimpleImputer(strategy="median")
train_numeric = imputer.fit_transform(X_train[NUM_CLASS])
val_numeric = imputer.transform(X_val[NUM_CLASS])
assert np.isfinite(train_numeric).all()
assert np.isfinite(val_numeric).all()
expected = X_train["monthly_fee"].median()
assert np.isclose(imputer.statistics_[NUM_CLASS.index("monthly_fee")], expected)
print(imputer.statistics_)

## 11장. 이상치와 잘못된 데이터 처리하기

불가능한 값과 드문 정상값을 구분하고 자료형을 변환한다.

In [ ]:
import numpy as np
from lab import load_customers, clean_customers, split_task

raw = load_customers()
cleaned = clean_customers(raw)
assert raw.loc[0, "monthly_fee"] == -10
assert np.isnan(cleaned.loc[0, "monthly_fee"])
assert cleaned.loc[2, "monthly_fee"] == 10000
X_train, X_val, X_test, y_train, y_val, y_test = split_task()
q1, q3 = X_train["monthly_fee"].quantile([0.25, 0.75])
iqr = q3-q1
lower, upper = q1-1.5*iqr, q3+1.5*iqr
flag = ~X_train["monthly_fee"].between(lower, upper) & X_train["monthly_fee"].notna()
print("Investigation candidates:", int(flag.sum()), "bounds:", lower, upper)

## 12장. 범주형 변수 인코딩하기

라벨·원핫 인코딩과 새로운 범주를 처리한다.

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

train = np.array([["monthly"], ["yearly"], ["monthly"]])
new = np.array([["trial"]])
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(train)
unknown = encoder.transform(new)
assert encoded.shape == (3, 2)
assert unknown.shape == (1, 2)
assert unknown.sum() == 0
labels = LabelEncoder().fit_transform(["stay", "leave", "stay"])
assert len(set(labels)) == 2
print(encoder.get_feature_names_out(), encoded, labels)

## 13장. 훈련·검증·테스트 데이터 나누기

입력·타깃·분할·층화·시간 순서를 구분한다.

In [ ]:
from sklearn.model_selection import train_test_split
from lab import load_customers, clean_customers, NUM_CLASS, CAT_CLASS

df = clean_customers(load_customers())
X = df[NUM_CLASS+CAT_CLASS]
y = df["churn"]
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev)
assert (len(X_train), len(X_val), len(X_test)) == (360, 120, 120)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index)
print(len(X_train), len(X_val), len(X_test))

## 14장. 스케일링과 데이터 누수 방지하기

훈련에서 학습한 변환을 검증·테스트에 적용한다.

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from lab import split_task, NUM_CLASS

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
numeric = Pipeline([("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())])
train = numeric.fit_transform(X_train[NUM_CLASS])
val = numeric.transform(X_val[NUM_CLASS])
assert np.allclose(train.mean(axis=0), 0, atol=1e-6)
assert train.shape[1] == val.shape[1] == 4
print("Train means:", train.mean(axis=0))
print("Validation means:", val.mean(axis=0))

## 15장. 전처리 흐름 완성하기

열 구성과 변환을 일관되게 유지하는 파이프라인을 만든다.

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
model = Pipeline([("prep", preprocessor()),
                  ("clf", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
prediction = model.predict(X_val)
features = model.named_steps["prep"].get_feature_names_out()
assert len(prediction) == len(y_val)
assert len(features) == model.named_steps["clf"].n_features_in_
assert np.isfinite(model.named_steps["prep"].transform(X_val)).all()
print(features)